<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/04_Foundations/statistics/05_experimental_design.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experimental Design

Design experiments that actually answer your questions — from A/B tests to multi-armed bandits, get causation not just correlation.

**Topics:** A/B test design, randomization, stratification, ANOVA, factorial designs, interaction effects, multi-armed bandits, Bayesian A/B testing

## 1. A/B Test Design — From Hypothesis to Sample Size

In [ ]:
import numpy as np
from scipy import stats

def ab_test_sample_size(
    baseline_rate: float,
    min_detectable_effect: float,  # absolute lift, e.g., 0.02 for +2%
    alpha: float = 0.05,
    power: float = 0.80,
    two_tailed: bool = True,
) -> dict:
    """Calculate sample size for proportions A/B test."""
    p1 = baseline_rate
    p2 = baseline_rate + min_detectable_effect

    # Pooled proportion under H0
    p_pool = (p1 + p2) / 2

    z_alpha = stats.norm.ppf(1 - alpha / (2 if two_tailed else 1))
    z_beta = stats.norm.ppf(power)

    # Standard formula for two proportions
    n = ((z_alpha * np.sqrt(2 * p_pool * (1 - p_pool)) +
           z_beta * np.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) /
          (p2 - p1)) ** 2

    return {
        'n_per_group': int(np.ceil(n)),
        'total_n': int(np.ceil(n)) * 2,
        'baseline': p1,
        'treatment': p2,
        'relative_lift': (p2 - p1) / p1,
        'alpha': alpha,
        'power': power,
    }

# Example: e-commerce checkout conversion
print('A/B test sample size calculator — Checkout conversion (baseline 8%)')
print(f'{"MDE":>8} {"Relative lift":>15} {"n per group":>14} {"Total users":>13} {"At 10k/day"}')
print('-' * 70)
for mde in [0.005, 0.01, 0.02, 0.04]:
    result = ab_test_sample_size(0.08, mde)
    days = result['total_n'] / 10_000
    print(f'{mde:>+8.1%} {result["relative_lift"]:>15.1%} {result["n_per_group"]:>14,} {result["total_n"]:>13,} {days:>6.1f} days')

print()
print('A/B test design checklist:')
checklist = [
    'Define ONE primary metric before launching',
    'Calculate sample size before collecting data (not after!)',
    'Randomize at the right unit (user-level, not session-level for most features)',
    'Check for novelty effect — run for at least 1-2 full business cycles',
    'Validate randomization with A/A test on historical data',
    'Pre-register your hypothesis and analysis plan',
    'Avoid peeking: only analyze after predetermined sample size is reached',
]
for item in checklist:
    print(f'  ☐ {item}')

## 2. Randomization and Stratification

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(42)

# Simulate user base with confounding variable (device type)
n_users = 2000
# 70% mobile, 30% desktop — mobile users convert less
device = np.random.choice(['mobile', 'desktop'], n_users, p=[0.7, 0.3])
baseline_rate = np.where(device == 'mobile', 0.04, 0.12)  # mobile converts at 4%, desktop 12%

def simple_randomize(users: np.ndarray) -> np.ndarray:
    """Completely random assignment."""
    return np.random.binomial(1, 0.5, len(users))

def stratified_randomize(device: np.ndarray) -> np.ndarray:
    """Stratified randomization: ensure 50/50 within each stratum."""
    assignment = np.zeros(len(device), dtype=int)
    for stratum in np.unique(device):
        mask = device == stratum
        n_stratum = mask.sum()
        assignment[mask] = np.random.permutation(
            [0] * (n_stratum // 2) + [1] * (n_stratum - n_stratum // 2)
        )
    return assignment

# Compare balance between methods
n_simulations = 1000
simple_imbalances = []
strat_imbalances = []

for _ in range(n_simulations):
    s_assign = simple_randomize(device)
    t_assign = stratified_randomize(device)

    # Measure imbalance in mobile rate between control/treatment
    for assign, imbalances in [(s_assign, simple_imbalances), (t_assign, strat_imbalances)]:
        control_mobile = (device[assign == 0] == 'mobile').mean()
        treat_mobile = (device[assign == 1] == 'mobile').mean()
        imbalances.append(abs(control_mobile - treat_mobile))

print('Randomization quality — mobile user imbalance between groups:')
print(f'  Simple randomization: mean imbalance = {np.mean(simple_imbalances):.4f}, std = {np.std(simple_imbalances):.4f}')
print(f'  Stratified:          mean imbalance = {np.mean(strat_imbalances):.4f}, std = {np.std(strat_imbalances):.4f}')
print(f'  → Stratification reduces imbalance variance by {(1 - np.std(strat_imbalances)/np.std(simple_imbalances))*100:.1f}%')
print()

# CUPED: Controlled-experiment Using Pre-Experiment Data
# Reduces variance by conditioning on pre-experiment metric
print('CUPED variance reduction:')
pre_revenue = np.random.lognormal(3, 0.8, n_users)  # pre-experiment revenue
treatment = np.random.binomial(1, 0.5, n_users)
# Post-experiment revenue: treatment lifts by 5%, correlated with pre
post_revenue = 0.7 * pre_revenue + np.random.normal(0, 5, n_users) + treatment * 2

# Standard difference-in-means
diff_raw = post_revenue[treatment==1].mean() - post_revenue[treatment==0].mean()
se_raw = np.sqrt(post_revenue[treatment==1].var()/treatment.sum() +
                  post_revenue[treatment==0].var()/(1-treatment).sum())

# CUPED: residualize out pre-experiment covariate
theta = np.cov(post_revenue, pre_revenue)[0,1] / np.var(pre_revenue)
cuped_post = post_revenue - theta * (pre_revenue - pre_revenue.mean())
diff_cuped = cuped_post[treatment==1].mean() - cuped_post[treatment==0].mean()
se_cuped = np.sqrt(cuped_post[treatment==1].var()/treatment.sum() +
                    cuped_post[treatment==0].var()/(1-treatment).sum())

print(f'  Raw:   estimate={diff_raw:.3f}, SE={se_raw:.3f}')
print(f'  CUPED: estimate={diff_cuped:.3f}, SE={se_cuped:.3f}')
print(f'  Variance reduction: {(1 - se_cuped**2/se_raw**2)*100:.1f}%  (equivalent to {se_raw**2/se_cuped**2:.1f}x more data!)')

## 3. ANOVA — Comparing Multiple Groups

In [ ]:
import numpy as np
from scipy import stats

np.random.seed(42)

# One-way ANOVA: test 4 ad variants simultaneously
variants = {
    'Control':   np.random.normal(5.2, 1.5, 100),  # CTR %
    'Variant A': np.random.normal(5.8, 1.4, 100),
    'Variant B': np.random.normal(6.1, 1.6, 100),
    'Variant C': np.random.normal(5.3, 1.5, 100),
}

# ANOVA F-test
f_stat, p_val = stats.f_oneway(*variants.values())
print('One-way ANOVA: 4 ad variants → CTR')
print(f'  F={f_stat:.3f}, p={p_val:.4f} → {"At least one variant differs" if p_val < 0.05 else "No significant differences"}')
print()

# ANOVA decomposition by hand
all_data = np.concatenate(list(variants.values()))
grand_mean = all_data.mean()
k = len(variants)   # groups
N = len(all_data)   # total observations
n_per_group = N // k

ss_between = sum(n_per_group * (g.mean() - grand_mean)**2 for g in variants.values())
ss_within = sum(np.sum((g - g.mean())**2) for g in variants.values())
ss_total = np.sum((all_data - grand_mean)**2)

df_between = k - 1
df_within = N - k
ms_between = ss_between / df_between
ms_within = ss_within / df_within
eta_squared = ss_between / ss_total  # effect size

print('ANOVA decomposition:')
print(f'  SS_between (treatment) = {ss_between:.2f}, df={df_between}, MS={ms_between:.3f}')
print(f'  SS_within  (error)     = {ss_within:.2f}, df={df_within}, MS={ms_within:.3f}')
print(f'  F = MS_between / MS_within = {ms_between/ms_within:.3f}')
print(f'  η² (eta-squared) = {eta_squared:.3f}  (effect size: proportion of variance explained)')
print()

# Post-hoc tests: which pairs differ? (Tukey HSD)
print('Post-hoc Tukey HSD pairwise comparisons (after significant ANOVA):')
group_names = list(variants.keys())
group_means = [g.mean() for g in variants.values()]
group_ns = [len(g) for g in variants.values()]

# Tukey critical value (approximate via studentized range)
q_crit = 3.68  # studentized range q(k=4, df=N-k=396, α=0.05)
se_diff = np.sqrt(ms_within * (1/n_per_group + 1/n_per_group))
hsd = q_crit * np.sqrt(ms_within / n_per_group)

for i in range(len(group_names)):
    for j in range(i+1, len(group_names)):
        diff = abs(group_means[i] - group_means[j])
        significant = diff > hsd
        print(f'  {group_names[i]} vs {group_names[j]}: diff={diff:.3f}, HSD={hsd:.3f} → {"SIGNIFICANT" if significant else "not significant"}')

## 4. Factorial Design and Interaction Effects

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(42)

# 2x2 factorial design: button_color × button_text → conversion
# Interaction effect: red button ONLY works with urgent text
n_per_cell = 500

conditions = {
    ('blue',  'normal'): 0.08,    # baseline
    ('blue',  'urgent'): 0.09,    # text effect: +1%
    ('red',   'normal'): 0.085,   # color effect: +0.5%
    ('red',   'urgent'): 0.14,    # INTERACTION: red + urgent = big lift
}

results = []
for (color, text), rate in conditions.items():
    conversions = np.random.binomial(1, rate, n_per_cell)
    results.append({'color': color, 'text': text,
                    'n': n_per_cell, 'conversions': conversions.sum(),
                    'observed_rate': conversions.mean(), 'true_rate': rate})

df = pd.DataFrame(results)
print('2×2 Factorial Design: Button Color × Text Style → Conversion')
print(df.to_string(index=False))
print()

# Interaction effect analysis
blue_normal = df.loc[(df.color=='blue') & (df.text=='normal'), 'observed_rate'].values[0]
blue_urgent = df.loc[(df.color=='blue') & (df.text=='urgent'), 'observed_rate'].values[0]
red_normal  = df.loc[(df.color=='red')  & (df.text=='normal'), 'observed_rate'].values[0]
red_urgent  = df.loc[(df.color=='red')  & (df.text=='urgent'), 'observed_rate'].values[0]

main_color = ((red_normal + red_urgent) - (blue_normal + blue_urgent)) / 2
main_text  = ((blue_urgent + red_urgent) - (blue_normal + red_normal)) / 2
interaction = (red_urgent - red_normal) - (blue_urgent - blue_normal)

print('Effect decomposition:')
print(f'  Main effect of color:  {main_color:+.4f}  ({main_color*100:+.2f}% avg lift from red)')
print(f'  Main effect of text:   {main_text:+.4f}  ({main_text*100:+.2f}% avg lift from urgent)')
print(f'  INTERACTION:           {interaction:+.4f}  ({interaction*100:+.2f}% extra when red+urgent)')
print()
print('Key insight: Running separate A/B tests would MISS the interaction!')
print('Factorial designs test BOTH main effects AND interactions simultaneously.')
print()
print('Why interactions matter in ML:')
print('  - Features can have synergistic effects (both features needed for the lift)')
print('  - Models without interaction terms will UNDERFIT when interactions exist')
print('  - Tree-based models learn interactions automatically')
print('  - Linear models need explicit x1*x2 interaction terms')

## 5. Bayesian A/B Testing and Multi-Armed Bandits

In [ ]:
import numpy as np
from scipy import stats

np.random.seed(42)

# Bayesian A/B test: Beta-Binomial conjugate model
class BayesianABTest:
    """Beta-Binomial model for conversion rate A/B tests."""
    def __init__(self, prior_alpha: float = 1.0, prior_beta: float = 1.0):
        self.alpha = prior_alpha  # pseudo successes
        self.beta = prior_beta    # pseudo failures

    def update(self, successes: int, trials: int) -> 'BayesianABTest':
        """Bayesian update: Beta(α + successes, β + failures)."""
        clone = BayesianABTest(self.alpha + successes, self.beta + (trials - successes))
        return clone

    def sample(self, n: int = 100_000) -> np.ndarray:
        return np.random.beta(self.alpha, self.beta, n)

    def mean(self) -> float:
        return self.alpha / (self.alpha + self.beta)

    def credible_interval(self, level: float = 0.95) -> tuple[float, float]:
        dist = stats.beta(self.alpha, self.beta)
        return dist.ppf((1-level)/2), dist.ppf(1-(1-level)/2)

# Simulate sequential data collection
true_rates = {'control': 0.08, 'treatment': 0.10}  # 25% relative lift
control_model = BayesianABTest(prior_alpha=2, prior_beta=20)  # weak prior ~9%
treatment_model = BayesianABTest(prior_alpha=2, prior_beta=20)

print('Bayesian A/B test — sequential updating')
print(f'{"n (per group)":<16} {"Control mean":<16} {"Treatment mean":<16} {"P(B>A)":<10} {"Conclusion"}')
print('-' * 75)

checkpoints = [50, 200, 500, 1000, 2000]
prev_n = 0
for n_total in checkpoints:
    new_n = n_total - prev_n
    # Observe new data
    ctrl_obs = np.random.binomial(new_n, true_rates['control'])
    trt_obs  = np.random.binomial(new_n, true_rates['treatment'])
    # Update posteriors
    control_model = control_model.update(ctrl_obs, new_n)
    treatment_model = treatment_model.update(trt_obs, new_n)
    # P(treatment > control) via Monte Carlo
    p_b_gt_a = (treatment_model.sample() > control_model.sample()).mean()
    conclusion = 'SHIP IT' if p_b_gt_a > 0.95 else 'REJECT' if p_b_gt_a < 0.05 else 'collect more'
    print(f'{n_total:<16} {control_model.mean():<16.4f} {treatment_model.mean():<16.4f} {p_b_gt_a:<10.4f} {conclusion}')
    prev_n = n_total

print()

# Thompson Sampling: multi-armed bandit
class ThompsonSampler:
    """Thompson sampling bandit — balances explore/exploit."""
    def __init__(self, n_arms: int):
        self.alphas = np.ones(n_arms)
        self.betas = np.ones(n_arms)

    def select_arm(self) -> int:
        samples = np.random.beta(self.alphas, self.betas)
        return int(np.argmax(samples))

    def update(self, arm: int, reward: int) -> None:
        self.alphas[arm] += reward
        self.betas[arm] += 1 - reward

true_arm_rates = [0.05, 0.08, 0.10, 0.07]  # arm 2 is best
bandit = ThompsonSampler(len(true_arm_rates))
arm_counts = np.zeros(len(true_arm_rates), dtype=int)
total_reward = 0

n_rounds = 10_000
for _ in range(n_rounds):
    arm = bandit.select_arm()
    reward = np.random.binomial(1, true_arm_rates[arm])
    bandit.update(arm, reward)
    arm_counts[arm] += 1
    total_reward += reward

optimal_reward = max(true_arm_rates) * n_rounds
regret = optimal_reward - total_reward

print('Thompson Sampling bandit (10,000 rounds):')
print(f'{"Arm":<6} {"True rate":<12} {"Times chosen":<15} {"% chosen"}')
for i, (rate, count) in enumerate(zip(true_arm_rates, arm_counts)):
    marker = '<-- BEST' if rate == max(true_arm_rates) else ''
    print(f'{i:<6} {rate:<12.2%} {count:<15,} {count/n_rounds:<10.1%} {marker}')
print(f'Total regret: {regret:.0f} ({regret/n_rounds:.2%} rate)  — vs {(1/4)*n_rounds:.0f} for pure explore')

## Exercises

1. **Peeking problem simulation:** Run 10,000 simulated A/B tests where H0 is TRUE (no real effect). For each, check significance every 50 observations until n=2000. What fraction of tests incorrectly conclude significance (false positive rate)? Compare to running only at the end. Implement sequential testing with the alpha-spending function to fix this.

2. **CUPED implementation:** Take the Ames Housing dataset. Randomly assign houses to treatment (add 5% to sale price). Compute pre-experiment variance explained by house square footage. Apply CUPED correction. Show how many fewer observations you need to achieve the same 80% power compared to naive difference-in-means.

3. **Bandit vs A/B comparison:** Simulate 50,000 users with 4 arms where arm 2 converts at 12% and others at 8%. Compare cumulative reward for (a) pure A/B test — equal allocation until n=10,000 then all traffic to winner, and (b) Thompson sampling throughout. What is the total revenue difference? When does the bandit become worse?